# 문화누리카드 수요모형 — 조원 인계용

이 노트북은 **최종 채택 모델인 `Plackett–Luce (Rank-Ordered Logit) + rank scale`만** 남긴 실행용 버전입니다.

### 조원이 해야 할 일
1. 국민여가활동조사 Excel 파일을 이 노트북과 같은 폴더에 둡니다.
2. 아래 셀을 **위에서부터 순서대로 실행**합니다.
3. `model.fit(...)`이 끝나면 `model.predict_proba(...)`로 10개 문화분류 선호확률을 바로 얻을 수 있습니다.

### 이 노트북에서 제거한 것
- 다항로짓·HGB 비교
- 3-fold 교차검증
- Leave-Seoul-out 검증
- VIF/GVIF 점검
- 트리 피처 중요도
- 시각화 및 모델 선정용 실험

위 항목은 이미 모델 선정 과정에서 사용한 검증/비교 작업이므로 **최종 모델을 다시 학습하고 사용하는 데 필요하지 않습니다.**

### 최종 출력
개인 \(i\)가 분류 \(k\)를 1순위로 선택할 확률

\[
\hat p_{ik}
=
\frac{\exp(z_{ik})}
{\sum_{\ell}\exp(z_{i\ell})}
\]

이 확률을 이후 100m 격자의 인구구성에 가중합하면 분류별 격자 수요를 만들 수 있습니다.

## 1. 환경 설정

필요 패키지는 `numpy`, `pandas`, `scipy`, `openpyxl`뿐입니다.

- `L2=1.0`: 희소 분류의 계수가 과도하게 커지는 것을 막는 ridge 정규화
- `GAMMA=0.7`: 2021~2025 자료에서 최근 연도에 더 큰 가중을 주는 연도 감쇠계수
- 데이터 파일명이 기존과 같으면 자동으로 찾습니다. 다른 이름이라면 `DATA_PATH` 한 줄만 수정하면 됩니다.

In [ ]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# -----------------------------
# 사용자가 바꿀 수 있는 설정
# -----------------------------
L2 = 1.0
GAMMA = 0.7
MAX_RANK = 3
SEED = 0
np.random.seed(SEED)

SHEET = "중분류_집계데이터"
DEFAULT_FILENAME = "국민여가활동조사_2021_2025_중분류별활동수_집계_원본값유지최종.xlsx"

# 같은 폴더의 기존 파일명을 우선 사용하고, 없으면 유사 파일명을 자동 탐색
DATA_PATH = Path(DEFAULT_FILENAME)
if not DATA_PATH.exists():
    candidates = sorted(Path(".").glob("국민여가활동조사*2021*2025*.xlsx"))
    if candidates:
        DATA_PATH = candidates[0]

print("DATA_PATH =", DATA_PATH)

## 2. 데이터 로드

목표변수는 국민여가활동조사의 **가장 만족스러운 여가활동 1·2·3순위**입니다.

원자료의 1~88 활동코드는 그대로 모델에 넣지 않고, 다음 단계에서 문화누리카드 접근성 분석에 맞는 **10개 대안**으로 묶습니다.

In [ ]:
R1 = "가장 만족스러운 여가활동 1순위(국민여가활동조사의 여가활동 코드(1~88)를 의미)"
R2 = "가장 만족스러운 여가활동 2순위(국민여가활동조사의 여가활동 코드(1~88)를 의미)"
R3 = "가장 만족스러운 여가활동 3순위(국민여가활동조사의 여가활동 코드(1~88)를 의미)"

DEMO_COLS = ["성별", "연령", "학력", "가구소득", "지역규모", "17개 시도", "장애여부"]
CONS_ACCESS_RAW = "여가활동 제약요인_여가시설 접근성 부족"
CONS_EXP_RAW = "여가활동 제약요인_여가활동 경험 부족"

REQUIRED_COLS = [
    R1, R2, R3,
    *DEMO_COLS,
    CONS_ACCESS_RAW, CONS_EXP_RAW,
    "최종가중치", "조사년도"
]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {DATA_PATH}\n"
        "Excel 파일을 노트북과 같은 폴더에 두거나 DATA_PATH를 수정하세요."
    )

df = pd.read_excel(DATA_PATH, sheet_name=SHEET)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise KeyError(f"필수 컬럼이 없습니다: {missing}")

print("데이터 shape:", df.shape)
print("1/2/3순위 응답 수:", df[R1].notna().sum(), df[R2].notna().sum(), df[R3].notna().sum())

## 3. 88개 활동코드 → 10개 대안

Plackett–Luce가 실제로 학습하는 선택지는 아래 10개입니다.

**시설문화 8개**  
도서 · 공연 · 미술 · 문화체험 · 관광 · 스포츠관람 · 체육시설 · 영화

**기타 2개**  
기타문화 · 비문화(outside option)

`비문화`를 제거하지 않는 이유는 문화활동을 선호하지 않는 사람까지 억지로 문화시설 수요에 배정하면 수요가 과대추정되기 때문입니다. 모델에서는 `비문화`를 마지막 기준대안으로 둡니다.

In [ ]:
CODE2ALT = {}

def register(codes, alt):
    for code in codes:
        CODE2ALT[code] = alt

register([65, 66, 78], "도서")
register([3, 4, 5, 6, 8], "공연")
register([1, 2, 14], "미술")
register([9, 10, 11, 12, 13, 15, 50, 51, 69], "문화체험")
register([38, 39, 40, 41, 43, 44, 45, 46, 47, 55, 72], "관광")
register([16, 18, 19], "스포츠관람")
register([20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 56], "체육시설")
register([7], "영화")

# 기타문화
register([17, 74, 75], "기타문화")
register([76, 77], "기타문화")
register([48], "기타문화")
register([42], "기타문화")
register([35], "기타문화")

# 문화누리 시설분류 밖의 활동
register([49, 52, 53, 54, 57, 58, 59, 60, 61, 62, 63, 64,
          67, 68, 70, 71, 73, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88], "비문화")

ALTS = [
    "도서", "공연", "미술", "문화체험", "관광",
    "스포츠관람", "체육시설", "영화", "기타문화", "비문화"
]
ALT_IDX = {alt: i for i, alt in enumerate(ALTS)}

assert ALTS[-1] == "비문화"
assert len(ALTS) == 10

print("대안:", ALTS)

## 4. 부분순위 라벨 만들기

원자료에는 1·2·3순위가 있지만, 88개 코드를 10개 대안으로 묶고 나면 같은 대안이 두 번 나타날 수 있습니다.

예를 들어 1순위와 2순위가 서로 다른 원래 활동코드라도 둘 다 `기타문화`로 매핑될 수 있습니다.  
Plackett–Luce는 한 순위열에서 같은 대안을 반복 선택할 수 없으므로 **처음 나온 대안만 남기고 중복은 제거**합니다.

따라서 각 응답자는 길이 1~3의 **부분순위(partial ranking)**를 갖습니다.

In [ ]:
def to_alt(v):
    if pd.isna(v):
        return None
    return CODE2ALT.get(int(v))

rankings = []

for r1, r2, r3 in zip(df[R1], df[R2], df[R3]):
    seq = []
    for value in (r1, r2, r3):
        alt = to_alt(value)
        if alt is None:
            continue

        j = ALT_IDX[alt]
        if j not in seq:
            seq.append(j)

    rankings.append(seq)

rank_len = pd.Series([len(s) for s in rankings])
print("부분순위 길이 분포")
print(rank_len.value_counts().sort_index())

## 5. 설명변수와 표본 가중치

모델의 입력 \(X\)는 다음으로 구성합니다.

- **인구특성 7개**: 성별, 연령, 학력, 가구소득, 지역규모, 17개 시도, 장애여부
- **통제변수 2개**: 시설 접근성 부족, 여가활동 경험 부족

범주형 인구특성은 one-hot encoding 합니다.

개인 \(i\)의 최종 학습 가중치는

\[
w_i
=
w_{survey,i}
\times
\gamma^{(2025-year_i)}
\]

로 만들고 평균이 1이 되도록 정규화합니다.

- `최종가중치`: 표본을 모집단 구조에 맞춤
- `GAMMA=0.7`: 코로나 시기의 오래된 관측값을 상대적으로 덜 반영

In [ ]:
def _to_code_string(series):
    """설문 범주코드를 학습/예측에서 동일한 문자열 형식으로 맞춤."""
    return series.astype("Int64").astype(str)

# 학습 데이터에서 사용된 전체 범주 순서를 저장
TRAIN_CATEGORIES = {
    col: sorted(_to_code_string(df[col]).unique().tolist())
    for col in DEMO_COLS
}

def make_feature_frame(raw_df, category_levels=TRAIN_CATEGORIES):
    """
    원자료 형태의 DataFrame을 모델 입력용 더미변수 DataFrame으로 변환.
    학습 때 정한 범주 순서를 고정해 새 데이터에서도 기준범주가 바뀌지 않게 한다.
    """
    demo_parts = []

    for col in DEMO_COLS:
        values = _to_code_string(raw_df[col])
        known = set(category_levels[col])
        unknown = sorted(set(values.unique()) - known)

        if unknown:
            raise ValueError(
                f"{col}에 학습 때 없던 범주가 있습니다: {unknown}"
            )

        cat = pd.Categorical(values, categories=category_levels[col])
        dummies = pd.get_dummies(
            cat,
            prefix=col,
            drop_first=True,
            dtype=float
        )
        dummies.index = raw_df.index
        demo_parts.append(dummies)

    demo_df = pd.concat(demo_parts, axis=1)

    cons_df = pd.DataFrame(
        {
            "cons_access": raw_df[CONS_ACCESS_RAW].notna().astype(int),
            "cons_exp": raw_df[CONS_EXP_RAW].notna().astype(int),
        },
        index=raw_df.index
    )

    return pd.concat([cons_df, demo_df], axis=1).astype(float)


Xdf = make_feature_frame(df)
X = Xdf.to_numpy(dtype=float)
FEATURE_NAMES = list(Xdf.columns)

# 조사 최종가중치 × 연도 감쇠
w_survey = df["최종가중치"].astype(float).to_numpy()
year = df["조사년도"].astype(int).to_numpy()
latest_year = year.max()

w_year = GAMMA ** (latest_year - year)
w = w_survey * w_year
w = w / w.mean()

print("X shape:", X.shape)
print("feature 수:", len(FEATURE_NAMES))
print("weight mean:", round(w.mean(), 6))

## 6. 최종 모델 — Plackett–Luce + rank scale

Plackett–Luce는 순위를 **남은 후보 중 하나를 순차적으로 선택하는 과정**으로 봅니다.

개인 \(i\)의 대안 \(k\)에 대한 선호점수는

\[
z_{ik} = \alpha_k + x_i^\top\beta_k
\]

이고, 각 순위 단계에서는 softmax로 선택확률을 계산합니다.

예를 들어 응답이 \(r_1 \rightarrow r_2 \rightarrow r_3\)라면,

\[
P(r_1,r_2,r_3)
=
\prod_t
\frac{\exp(z_{ir_t}/\lambda_t)}
{\sum_{k \in S_t}\exp(z_{ik}/\lambda_t)}
\]

입니다.

- \(S_t\): 그 단계에서 아직 선택되지 않은 대안 집합
- \(\lambda_1=1\): 1순위 기준
- \(\lambda_2,\lambda_3\): 2·3순위의 상대적 scale을 데이터가 학습
- `비문화`의 계수는 0으로 고정해 기준대안으로 사용
- L2 정규화를 적용해 희소 분류 계수를 안정화

아래 클래스 하나가 **학습(`fit`)과 확률예측(`predict_proba`)을 모두 담당**합니다.

In [ ]:
class RankScalePlackettLuce:
    def __init__(self, n_alternatives, l2=1.0, max_rank=3, maxiter=300):
        self.n_alternatives = int(n_alternatives)
        self.l2 = float(l2)
        self.max_rank = int(max_rank)
        self.maxiter = int(maxiter)

    def fit(self, X, rankings, sample_weight=None):
        X = np.asarray(X, dtype=float)
        n, p = X.shape
        A = self.n_alternatives
        FREE = A - 1

        if len(rankings) != n:
            raise ValueError("X 행 수와 rankings 길이가 다릅니다.")

        if sample_weight is None:
            sample_weight = np.ones(n, dtype=float)
        else:
            sample_weight = np.asarray(sample_weight, dtype=float)

        choice = np.full((n, self.max_rank), -1, dtype=np.int64)
        for i, seq in enumerate(rankings):
            for t, j in enumerate(seq[:self.max_rank]):
                choice[i, t] = j
        valid = choice >= 0

        n_base_params = FREE + FREE * p

        def unpack_base(theta):
            alpha = np.zeros(A)
            beta = np.zeros((A, p))

            alpha[:FREE] = theta[:FREE]
            beta[:FREE] = theta[FREE:n_base_params].reshape(FREE, p)
            return alpha, beta

        def core(theta, need_grad=True):
            alpha, beta = unpack_base(theta)

            rank_scale = np.ones(self.max_rank)
            if self.max_rank >= 2:
                rank_scale[1:] = np.exp(theta[n_base_params:])

            Z = alpha[None, :] + X @ beta.T

            grad_alpha = np.zeros(A)
            grad_beta = np.zeros((A, p))
            nll = 0.0
            available = np.ones((n, A), dtype=bool)

            for t in range(self.max_rank):
                v = valid[:, t]
                c = choice[:, t]
                lam = rank_scale[t]

                Z_scaled = Z / lam
                Zt = np.where(available, Z_scaled, -np.inf)

                m = Zt.max(axis=1, keepdims=True)
                exp_z = np.where(available, np.exp(Zt - m), 0.0)
                denom = exp_z.sum(axis=1, keepdims=True)
                probs = exp_z / denom

                log_denom = m.squeeze(1) + np.log(denom.squeeze(1))
                idx = np.where(v, c, 0)
                chosen_score = Z_scaled[np.arange(n), idx]

                nll += (
                    sample_weight
                    * np.where(v, -(chosen_score - log_denom), 0.0)
                ).sum()

                if need_grad:
                    G = probs.copy()
                    G[np.arange(n), idx] -= 1.0
                    G *= (sample_weight * v)[:, None] / lam

                    grad_alpha += G.sum(axis=0)
                    grad_beta += G.T @ X

                rows = np.arange(n)
                available[rows, idx] &= ~v

            nll += 0.5 * self.l2 * (
                (alpha[:FREE] ** 2).sum()
                + (beta[:FREE] ** 2).sum()
            )

            if not need_grad:
                return nll

            grad_alpha[:FREE] += self.l2 * alpha[:FREE]
            grad_beta[:FREE] += self.l2 * beta[:FREE]

            grad_base = np.concatenate([
                grad_alpha[:FREE],
                grad_beta[:FREE].ravel()
            ])

            return nll, grad_base

        def objective(theta):
            nll, grad_base = core(theta, need_grad=True)

            eps = 1e-4
            grad_scale = []

            for k in range(self.max_rank - 1):
                pos = n_base_params + k
                theta_plus = theta.copy()
                theta_plus[pos] += eps
                grad_scale.append((core(theta_plus, need_grad=False) - nll) / eps)

            grad = np.concatenate([grad_base, np.asarray(grad_scale)])
            return nll, grad

        theta0 = np.zeros(n_base_params + (self.max_rank - 1))

        t0 = time.time()
        result = minimize(
            objective,
            theta0,
            jac=True,
            method="L-BFGS-B",
            options={"maxiter": self.maxiter}
        )

        self.result_ = result
        self.intercept_, self.coef_ = unpack_base(result.x)

        self.rank_scale_ = np.ones(self.max_rank)
        if self.max_rank >= 2:
            self.rank_scale_[1:] = np.exp(result.x[n_base_params:])

        self.n_features_in_ = p
        self.fit_seconds_ = time.time() - t0
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)

        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                f"feature 수가 다릅니다. 학습={self.n_features_in_}, 입력={X.shape[1]}"
            )

        Z = self.intercept_[None, :] + X @ self.coef_.T
        Z = Z - Z.max(axis=1, keepdims=True)
        exp_z = np.exp(Z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

## 7. 모델 학습

실제로 모델을 돌리는 핵심은 아래 두 줄입니다.

```python
model = RankScalePlackettLuce(...)
model.fit(X, rankings, sample_weight=w)
```

학습이 끝나면 `model.intercept_`, `model.coef_`, `model.rank_scale_`에 추정 결과가 저장됩니다.

In [ ]:
model = RankScalePlackettLuce(
    n_alternatives=len(ALTS),
    l2=L2,
    max_rank=MAX_RANK,
    maxiter=300
)

model.fit(X, rankings, sample_weight=w)

print("수렴:", model.result_.success)
print("iteration:", model.result_.nit)
print("학습시간(초):", round(model.fit_seconds_, 1))
print("rank scale:", np.round(model.rank_scale_, 3))

if not model.result_.success:
    print("주의:", model.result_.message)

## 8. 학습 결과 확인

이 셀은 새로운 모델을 비교하거나 튜닝하기 위한 것이 아니라, **인계받은 조원이 학습이 정상적으로 재현됐는지 확인하는 최소 점검**입니다.

기존 실행에서 대략 다음 수준이었습니다.

- `λ2 ≈ 0.785`, `λ3 ≈ 0.788`
- 1순위 weighted log-loss ≈ `1.721`
- ECE ≈ `0.024`
- Top-1 ≈ `0.331`
- Top-3 ≈ `0.728`

환경이나 최적화 종료점에 따라 마지막 자릿수는 조금 달라질 수 있습니다.

In [ ]:
P_full = model.predict_proba(X)
y1 = np.array([seq[0] if seq else -1 for seq in rankings])

if (y1 < 0).any():
    raise ValueError("1순위 대안으로 매핑되지 않은 응답자가 있습니다.")

def weighted_logloss(P, y, weight):
    p_true = np.clip(P[np.arange(len(y)), y], 1e-12, 1.0)
    return -(weight * np.log(p_true)).sum() / weight.sum()

def weighted_ece(P, y, weight, bins=10):
    confidence = P.max(axis=1)
    prediction = P.argmax(axis=1)
    correct = (prediction == y).astype(float)

    total_weight = weight.sum()
    ece = 0.0

    for b in range(bins):
        lo = b / bins
        hi = (b + 1) / bins

        if b == bins - 1:
            mask = (confidence >= lo) & (confidence <= hi)
        else:
            mask = (confidence >= lo) & (confidence < hi)

        if not mask.any():
            continue

        weighted_acc = (weight[mask] * correct[mask]).sum()
        weighted_conf = (weight[mask] * confidence[mask]).sum()
        ece += abs(weighted_acc - weighted_conf) / total_weight

    return ece

def weighted_topk(P, y, weight, k):
    topk = np.argpartition(-P, kth=k-1, axis=1)[:, :k]
    hit = np.array([y[i] in topk[i] for i in range(len(y))], dtype=float)
    return (weight * hit).sum() / weight.sum()

metrics = {
    "weighted_logloss": weighted_logloss(P_full, y1, w),
    "ECE": weighted_ece(P_full, y1, w),
    "Top-1": weighted_topk(P_full, y1, w, 1),
    "Top-3": weighted_topk(P_full, y1, w, 3),
}

print(pd.Series(metrics).round(4))
print("\n평균 선호확률:")
mean_pref = (P_full * w[:, None]).sum(axis=0) / w.sum()
print(pd.Series(mean_pref, index=ALTS).sort_values(ascending=False).round(4))

## 9. 새 데이터에 선호확률 예측

중요한 점은 **학습 때 만든 더미변수 열과 예측 때의 열을 정확히 맞추는 것**입니다.

아래 함수가 이 정렬을 자동으로 처리합니다.

새 데이터는 원자료와 같은 인구특성 컬럼을 갖고 있어야 합니다.  
통제변수(`시설 접근성 부족`, `여가활동 경험 부족`)도 최종 학습모형에 포함되어 있으므로 같은 형태로 제공하는 것이 원칙입니다.

In [ ]:
def transform_new_data(
    raw_df,
    feature_names=FEATURE_NAMES,
    category_levels=TRAIN_CATEGORIES
):
    """
    원자료 형태의 새 데이터를 학습 때와 동일한 설계행렬로 변환.
    학습 당시 범주 순서와 feature 열을 그대로 사용한다.
    """
    required_for_prediction = [
        *DEMO_COLS,
        CONS_ACCESS_RAW,
        CONS_EXP_RAW
    ]

    missing = [c for c in required_for_prediction if c not in raw_df.columns]
    if missing:
        raise KeyError(
            "예측 데이터에 필요한 컬럼이 없습니다: "
            + ", ".join(missing)
        )

    new_Xdf = make_feature_frame(
        raw_df,
        category_levels=category_levels
    )
    new_Xdf = new_Xdf.reindex(
        columns=feature_names,
        fill_value=0.0
    )
    return new_Xdf


# 사용 예시: 원자료 앞 5명을 다시 예측
example_raw = df.iloc[:5].copy()
example_Xdf = transform_new_data(example_raw)
example_P = model.predict_proba(example_Xdf.to_numpy())

example_result = pd.DataFrame(
    example_P,
    columns=ALTS,
    index=example_raw.index
)
display(example_result.round(4))

## 10. 모델 저장 — 이후 재학습 없이 사용

아래 셀까지 실행하면 학습된 계수와 전처리 정보를 `pl_rankscale_model.pkl`로 저장합니다.

다음 분석 단계에서 매번 5만 명 데이터를 다시 학습할 필요 없이 이 파일을 불러와 사용할 수 있습니다.

In [ ]:
MODEL_PATH = Path("pl_rankscale_model.pkl")

artifact = {
    "model_name": "Plackett-Luce + rank scale",
    "alternatives": ALTS,
    "feature_names": FEATURE_NAMES,
    "category_levels": TRAIN_CATEGORIES,
    "intercept": model.intercept_,
    "coef": model.coef_,
    "rank_scale": model.rank_scale_,
    "l2": L2,
    "gamma": GAMMA,
    "demo_cols": DEMO_COLS,
    "cons_access_raw": CONS_ACCESS_RAW,
    "cons_exp_raw": CONS_EXP_RAW,
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(artifact, f)

print("저장 완료:", MODEL_PATH.resolve())

## 11. 저장된 모델만 불러와 예측하기

이미 `pl_rankscale_model.pkl`이 있다면 **학습 셀을 다시 돌릴 필요가 없습니다.**

아래 함수는 저장된 계수만으로 softmax 선호확률을 계산합니다.  
즉, 이후 격자 수요 집계 담당자는 학습 과정 전체를 몰라도 `artifact + 새 X`만 있으면 됩니다.

In [ ]:
def load_artifact(path="pl_rankscale_model.pkl"):
    with open(path, "rb") as f:
        return pickle.load(f)

def transform_with_artifact(raw_df, artifact):
    """
    저장된 artifact만 이용해 새 원자료를 학습 때와 동일한 X로 변환.
    원래 학습 데이터(df)는 필요하지 않다.
    """
    demo_cols = artifact["demo_cols"]
    category_levels = artifact["category_levels"]
    access_col = artifact["cons_access_raw"]
    exp_col = artifact["cons_exp_raw"]

    required = [*demo_cols, access_col, exp_col]
    missing = [c for c in required if c not in raw_df.columns]
    if missing:
        raise KeyError(
            "예측 데이터에 필요한 컬럼이 없습니다: "
            + ", ".join(missing)
        )

    demo_parts = []

    for col in demo_cols:
        values = raw_df[col].astype("Int64").astype(str)
        known = set(category_levels[col])
        unknown = sorted(set(values.unique()) - known)

        if unknown:
            raise ValueError(
                f"{col}에 학습 때 없던 범주가 있습니다: {unknown}"
            )

        cat = pd.Categorical(
            values,
            categories=category_levels[col]
        )
        dummies = pd.get_dummies(
            cat,
            prefix=col,
            drop_first=True,
            dtype=float
        )
        dummies.index = raw_df.index
        demo_parts.append(dummies)

    demo_df = pd.concat(demo_parts, axis=1)

    cons_df = pd.DataFrame(
        {
            "cons_access": raw_df[access_col].notna().astype(int),
            "cons_exp": raw_df[exp_col].notna().astype(int),
        },
        index=raw_df.index
    )

    Xdf_new = pd.concat(
        [cons_df, demo_df],
        axis=1
    ).astype(float)

    return Xdf_new.reindex(
        columns=artifact["feature_names"],
        fill_value=0.0
    )

def predict_from_artifact(X_new, artifact):
    X_new = np.asarray(X_new, dtype=float)
    intercept = np.asarray(artifact["intercept"])
    coef = np.asarray(artifact["coef"])

    Z = intercept[None, :] + X_new @ coef.T
    Z = Z - Z.max(axis=1, keepdims=True)
    exp_z = np.exp(Z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


# 저장된 모델만 사용한 예시
loaded = load_artifact(MODEL_PATH)
example_X_loaded = transform_with_artifact(example_raw, loaded)
P_loaded = predict_from_artifact(
    example_X_loaded.to_numpy(),
    loaded
)

print(
    "저장 전/후 예측 최대 차이:",
    np.abs(P_loaded - example_P).max()
)

---

## 조원에게 전달할 핵심

- **학습모델:** Plackett–Luce + rank scale
- **입력:** 인구특성 7개 + 제약요인 2개
- **목표:** 가장 만족스러운 여가활동 1·2·3순위
- **출력:** 10개 대안별 개인 선호확률
- **실행:** 위에서부터 실행 → 7번에서 모델 학습 완료
- **재사용:** 10번에서 `pl_rankscale_model.pkl` 저장 → 이후 재학습 없이 사용 가능

### 다음 단계에서 필요한 값

격자 \(g\)의 분류 \(k\) 수요는 개념적으로

\[
\hat D_{gk}
=
\sum_h N_{gh}\hat p_{hk}
\]

처럼 계산합니다.

- \(h\): 격자 내 인구집단
- \(N_{gh}\): 격자 \(g\)의 집단 \(h\) 인구수
- \(\hat p_{hk}\): 이 모델이 예측한 집단 \(h\)의 분류 \(k\) 선호확률

이 노트북은 **개인/인구집단 선호확률을 만드는 단계까지만** 담당합니다. E2SFCA 및 접근성 계산 코드는 의도적으로 넣지 않았습니다.